# 🥇 05_ingestao_gold — Camada Gold de `ecommerce_pedidos`
**Squad 2 · Dupla 3** · Passo 7 do plano · KPIs das regras 6 a 9 e alertas das regras 5 a 10

Lê a Silver e, a cada ciclo do orquestrador, recalcula os KPIs com janela de tempo, publica os alertas e espelha tudo no SQL Server.
Como pede o Passo 7: **KPIs em `overwrite`** (a fotografia atual) e **históricos em `append`**.

| Regra | Critério da planilha | Tabela de KPI | Alerta |
|---|---|---|---|
| 6 | volume de pedidos nos últimos 5 min (janela deslizante) | `gold_kpi_volume_pedidos_5min` | abaixo de X pedidos/5 min no horário de pico (> 9h e < 23h); X = 3 |
| 7 | ticket médio por método de pagamento em tempo real | `gold_kpi_ticket_medio_pagamento` | Pix mais de 30% abaixo da sua média histórica |
| 8 | % de `CANCELADO` sobre o total | `gold_kpi_taxa_cancelamento_30min` | taxa > 10% em qualquer janela de 30 min |
| 9 | receita acumulada no dia (`valor_total` desde a meia-noite) | `gold_kpi_receita_dia` | fora de ±30% do mesmo dia da semana anterior |
| 5 e 10 | pedido reprovado na Silver (status fora da lista; atualização antes do pedido) | — | um alerta por micro-lote, no ciclo em que ele chega |

O X da regra 6 e o limite da regra 9 não estão na planilha: foram decididos com o Geovany.

| Tabelas | Modo |
|---|---|
| `gold_kpi_*` (4 KPIs) | `overwrite` |
| `gold_hist_*` (histórico de cada KPI, com `ciclo_id`) | `append` |
| `gold_alertas` (mural de alertas) | `append` |

Destino: `squad2/gold/ecommerce_pedidos/<tabela>` (Delta) e `squad2.<tabela>` (SQL Server). Horários dos KPIs em **Brasília**.

| Etapa | Conteúdo |
|---|---|
| 1 | Configuração, parâmetros e teste do SQL Server |
| 2 | Leitura da Silver e horário de referência |
| 3 | Regra 6 · volume em 5 minutos |
| 4 | Regra 7 · ticket médio por pagamento |
| 5 | Regra 8 · taxa de cancelamento em 30 minutos |
| 6 | Regra 9 · receita acumulada no dia |
| 7 | Alertas (regras 5 a 10) |
| 8 | Gravação Delta |
| 9 | Espelho no SQL Server |
| 10 | Resumo para o orquestrador |

Cada etapa **faz** o trabalho e **valida** em seguida: ✅ ok · ⚠️ achado (segue) · ❌ erro (para).

## Etapa 1 — Configuração
**Faz:** carrega o `00_setup_config`, os parâmetros das regras (os critérios da planilha, mais o X da regra 6 e o limite da regra 9 decididos com o Geovany;
o *watermark* vem da EDA) e testa o SQL Server. Se o banco não responder, a Gold grava a camada Delta normalmente e o espelho fica para o próximo ciclo.
**Valida:** Silver disponível, parâmetros coerentes e banco acessível.

In [0]:
VALIDACAO_COMPLETA = False

In [0]:
%run ./00_setup_config

# ⚙️ 00_setup_config — Configuração central
**Squad 2 · Dupla 3 · `ecommerce_pedidos`**

Prepara endereços, credenciais, conexões e funções usadas por todo o pipeline. **Não lê nem grava pedidos.**

Cada etapa **faz o trabalho e valida em seguida**: ✅ ok · ⚠️ aviso (segue) · ❌ erro (para e explica).

**Uso nos outros notebooks** (duas células; o `%run` fica sozinho):
```
VALIDACAO_COMPLETA = False
```
```
%run ./00_setup_config
```
Dependências em **Environment → Dependencies**: `azure-identity`, `azure-storage-file-datalake`.

## Etapa 1 — Ambiente e endereços
**Faz:** cria a função `checar` e guarda os nomes fixos do projeto.
**Valida:** se está no Databricks e se origem e destino são containers diferentes.

Modo de validação: RÁPIDA
✅ Rodando no Databricks
✅ Origem (raw) e destino (squad2) separados


## Etapa 2 — Credenciais
**Faz:** busca as 7 chaves no Secret Scope. Ficam só na memória (aparecem como `[REDACTED]` se impressas).
**Valida:** se o cofre existe, se nenhuma chave falta e se o host do SQL tem o formato certo.

✅ Cofre 'mercadata-ecommerce-pedidos' encontrado
✅ 7 credenciais carregadas (valores ocultos)
✅ SQL_HOST no formato do Azure SQL


...

✅ url_origem OK
✅ tabela_delta_existe OK (Bronze já existe)
✅ ler_metadata OK (None para anotação inexistente)


## Etapa 6 — Funções do SQL Server
**Faz:** cria as funções de gravar e consultar o banco. Gravação pelo conector `sqlserver` (o `jdbc` é bloqueado no Serverless); leitura pelo `jdbc`.
**Valida:** se toda tabela recebe o schema `squad2`.

✅ Tabelas SQL vão para o schema 'squad2'


## Etapa 7 — Teste completo (só ao rodar este notebook direto)
**Faz e valida:** varre o `raw` · lê o micro-lote mais recente e confere colunas e tipos (regra 1) · grava e relê em `metadata/` · `SELECT 1` no SQL Server · confere se o schema `squad2` existe no banco.
No modo rápido (`%run`), é pulada. No fim, limpa as variáveis temporárias.

Teste completo pulado (modo rápido).
⚙️ 00_setup_config pronto para uso


In [0]:
from pyspark.sql import functions as F

contrato_gold = (ler_metadata("eda_raw") or {}).get("gold", {})
PARAMETROS = {
    # Regra 6: janela deslizante de 5 min; alerta abaixo de X no pico "> 9h e < 23h" (Brasília). X = 3 decidido com o Geovany
    "janela_volume_min": 5, "x_regra6": 3, "pico_inicio_h": 9, "pico_fim_h": 23, "horas_placar_volume": 2,
    # Regra 7: ticket "em tempo real" = última hora; alerta se o Pix cair mais de 30% abaixo da sua média histórica
    "janela_ticket_min": 60, "queda_pix_pct": 30, "metodo_monitorado": "pix",
    # Regra 8: alerta se a taxa passar de 10% em qualquer janela de 30 min
    "janela_cancelamento_min": 30, "cancelamento_max_pct": 10, "horas_placar_cancelamento": 24,
    # Regra 9: receita desde a meia-noite contra o mesmo dia da semana anterior; ±30% decidido com o Geovany
    "variacao_receita_pct": 30, "dias_placar_receita": 14,
    # Streaming: uma janela só "fecha" depois do atraso máximo de chegada medido na EDA (até lá, pedido atrasado ainda entra nela)
    "watermark_s": int(contrato_gold.get("watermark_s") or 120),
}
STATUS_CANCELADO = "CANCELADO"
TABELAS = ["gold_kpi_volume_pedidos_5min", "gold_kpi_ticket_medio_pagamento", "gold_kpi_taxa_cancelamento_30min", "gold_kpi_receita_dia",
           "gold_hist_volume_pedidos_5min", "gold_hist_ticket_medio_pagamento", "gold_hist_taxa_cancelamento_30min", "gold_hist_receita_dia",
           "gold_alertas"]
ANOTACAO_CONTROLE = "gold_controle"                             # marca da quarentena já avisada (regras 5 e 10)
CICLO_EM = datetime.now(timezone.utc)
CICLO_ID = int(CICLO_EM.strftime("%Y%m%d%H%M%S"))               # identifica a rodada em todos os históricos


def razao(parte, total):
    """Divisão protegida: devolve None quando o total é zero ou vazio (o Serverless em modo ANSI quebra ao dividir por zero)."""
    return None if not total else parte / total


def para_spark(pdf):
    """pandas -> Spark; None para tabela vazia (o Spark não infere schema de DataFrame vazio).
    Coluna 100% vazia (ex.: sem semana anterior) vira decimal, porque o Spark não adivinha o tipo de uma coluna só com nulos;
    e NaN vira nulo de verdade, para o SQL Server receber NULL e não um 'não-número'."""
    if not len(pdf):
        return None
    vazias = {c: "float64" for c in pdf.columns if pdf[c].isna().all()}
    df = spark.createDataFrame(pdf.astype(vazias))
    decimais = {c for c, t in df.dtypes if t == "double"}
    return df.select(*[F.when(F.isnan(c), None).otherwise(F.col(c)).alias(c) if c in decimais else F.col(c) for c in df.columns])


# ------------------------------ validação ------------------------------
checar(tabela_delta_existe(CAMINHO_SILVER), "Silver encontrada", "Silver não existe: rode o 04_ingestao_silver antes")
checar(PARAMETROS["x_regra6"] >= 1 and PARAMETROS["pico_inicio_h"] < PARAMETROS["pico_fim_h"], "Parâmetros coerentes", "Parâmetros inválidos")
x_eda = contrato_gold.get("x_regra6")                           # a EDA segue calculando o X sugerido: se mudar, vale rever com o negócio
checar(x_eda in (None, PARAMETROS["x_regra6"]), f"Regra 6 com X = {PARAMETROS['x_regra6']} (decisão do negócio)",
       f"A EDA sugere X = {x_eda}, diferente do X decidido ({PARAMETROS['x_regra6']}); vale levar ao Geovany", parar=False)
try:
    SQL_DISPONIVEL, _erro = ler_sql_query("SELECT 1 AS ok").first()["ok"] == 1, None
except Exception as erro:
    SQL_DISPONIVEL, _erro = False, str(erro)[:150]
checar(SQL_DISPONIVEL, "SQL Server respondeu", f"SQL Server indisponível ({_erro}): Delta será gravada e o espelho fica para o próximo ciclo", parar=False)
print(f"ciclo_id {CICLO_ID} · X = {PARAMETROS['x_regra6']} · watermark = {PARAMETROS['watermark_s']} s")

✅ Silver encontrada
✅ Parâmetros coerentes
✅ Regra 6 com X = 3 (decisão do negócio)
✅ SQL Server respondeu
ciclo_id 20261008132131 · X = 3 · watermark = 120 s


## Etapa 2 — Leitura da Silver e horário de referência
**Faz:** lê a Silver (datas em UTC) e cria `dt_local` no horário de Brasília. O "agora" dos KPIs é o **pedido mais recente** (tempo do evento, como no
Structured Streaming), e não o relógio: assim, um micro-lote que chega atrasado cai na janela certa e o resultado não depende de quando o ciclo rodou.
**Valida:** se há pedidos na Silver.

In [0]:
base = (ler_delta(CAMINHO_SILVER)
        .select("id_pedido", (F.col("dt_pedido") - F.expr("INTERVAL 3 HOURS")).alias("dt_local"),   # UTC -> Brasília
                "status_pedido", F.col("valor_total").cast("double").alias("valor_total"), "metodo_pagamento"))
geral = base.agg(F.count("*").alias("pedidos"), F.max("dt_local").alias("referencia")).first()
REFERENCIA = pd.Timestamp(geral["referencia"]) if geral["referencia"] else None
WATERMARK = pd.Timedelta(seconds=PARAMETROS["watermark_s"])
agora_local = pd.Timestamp(CICLO_EM).tz_convert(None) - pd.Timedelta(hours=3)

# ------------------------------ validação ------------------------------
checar(geral["pedidos"] > 0, f"{geral['pedidos']:,} pedidos na Silver", "Silver vazia: nada para calcular")
print(f"Referência (pedido mais recente, Brasília): {REFERENCIA} · idade: {(agora_local - REFERENCIA).total_seconds() / 60:.0f} min")

✅ 1,123 pedidos na Silver
Referência (pedido mais recente, Brasília): 2026-10-07 10:55:55.051961 · idade: 1406 min


## Etapa 3 — Regra 6 · volume de pedidos nos últimos 5 minutos
**Faz:** conta os pedidos por minuto e soma de 5 em 5, andando de minuto em minuto (**janela deslizante**), inclusive minutos sem pedido.
Situação de cada janela: `aberta` (ainda dentro do *watermark*: pode receber pedido atrasado), `fora_do_pico` (a janela não está inteira entre 9h e 23h),
`alerta` (menos de X pedidos) ou `normal`. Como na planilha, minuto sem pedido conta zero: se o gerador parar no pico, isso aparece como queda de volume.
**Valida:** a soma por minuto bate com a contagem da Silver no mesmo período.

In [0]:
JANELA_V = pd.Timedelta(minutes=PARAMETROS["janela_volume_min"])
inicio_placar = REFERENCIA.floor("min") - pd.Timedelta(hours=PARAMETROS["horas_placar_volume"])
inicio_contagem = inicio_placar - JANELA_V                     # a primeira janela do placar precisa dos minutos anteriores

por_minuto = (base.filter(F.col("dt_local") >= F.lit(inicio_contagem.to_pydatetime()))
              .groupBy(F.date_trunc("minute", "dt_local").alias("minuto")).count().toPandas())
grade = (pd.Series(0, index=pd.date_range(inicio_contagem, REFERENCIA.floor("min"), freq="min"))
         .add(por_minuto.set_index("minuto")["count"], fill_value=0))

# Janela que termina no minuto t: soma dos 5 minutos até t (inclusive)
volume = grade.rolling(PARAMETROS["janela_volume_min"]).sum().dropna().rename("pedidos").reset_index().rename(columns={"index": "minuto"})
volume["janela_inicio"] = volume["minuto"] - JANELA_V + pd.Timedelta(minutes=1)
volume["janela_fim"] = volume["minuto"] + pd.Timedelta(minutes=1)
volume = volume[volume["janela_inicio"] >= inicio_placar].drop(columns="minuto")
volume["pedidos"] = volume["pedidos"].astype(int)
# Pico "> 9h e < 23h": a janela inteira dentro do mesmo dia, de 9h00 até 23h00
volume["no_pico"] = (volume["janela_inicio"].dt.hour >= PARAMETROS["pico_inicio_h"]) & (
    volume["janela_fim"] <= volume["janela_inicio"].dt.normalize() + pd.Timedelta(hours=PARAMETROS["pico_fim_h"]))
volume["fechada"] = volume["janela_fim"] <= REFERENCIA - WATERMARK
volume["limite_x"] = PARAMETROS["x_regra6"]
volume["situacao"] = [("aberta" if not f else "fora_do_pico" if not p else "alerta" if n < PARAMETROS["x_regra6"] else "normal")
                      for f, p, n in zip(volume["fechada"], volume["no_pico"], volume["pedidos"])]

# ------------------------------ validação ------------------------------
silver_periodo = base.filter(F.col("dt_local") >= F.lit(inicio_contagem.to_pydatetime())).count()
checar(int(grade.sum()) == silver_periodo, f"{silver_periodo} pedidos no período, iguais aos da Silver", "Contagem por minuto diverge da Silver")

# ------------------------------ resultado ------------------------------
display(volume.sort_values("janela_inicio", ascending=False).head(20))
print(volume["situacao"].value_counts().to_dict())

✅ 203 pedidos no período, iguais aos da Silver


pedidos,janela_inicio,janela_fim,no_pico,fechada,limite_x,situacao
9,2026-10-07T10:51:00.000Z,2026-10-07T10:56:00.000Z,true,false,3,aberta
4,2026-10-07T10:50:00.000Z,2026-10-07T10:55:00.000Z,true,false,3,aberta
4,2026-10-07T10:49:00.000Z,2026-10-07T10:54:00.000Z,true,false,3,aberta
2,2026-10-07T10:48:00.000Z,2026-10-07T10:53:00.000Z,true,true,3,alerta
12,2026-10-07T10:47:00.000Z,2026-10-07T10:52:00.000Z,true,true,3,normal
11,2026-10-07T10:46:00.000Z,2026-10-07T10:51:00.000Z,true,true,3,normal
11,2026-10-07T10:45:00.000Z,2026-10-07T10:50:00.000Z,true,true,3,normal
11,2026-10-07T10:44:00.000Z,2026-10-07T10:49:00.000Z,true,true,3,normal
14,2026-10-07T10:43:00.000Z,2026-10-07T10:48:00.000Z,true,true,3,normal
3,2026-10-07T10:42:00.000Z,2026-10-07T10:47:00.000Z,true,true,3,normal


{'normal': 91, 'alerta': 18, 'fora_do_pico': 5, 'aberta': 3}


## Etapa 4 — Regra 7 · ticket médio por método de pagamento
**Faz:** ticket de cada método na janela recente (última hora até o pedido de referência) e a **média histórica** do mesmo método com os pedidos anteriores
à janela, ambos da Silver. A planilha só define alerta para o **Pix**: `alerta` se ele ficar **mais de** 30% abaixo da sua média histórica.
Os outros métodos aparecem no KPI como `nao_monitorado`.
**Valida:** ticket × pedidos = receita de cada método na janela.

In [0]:
inicio_ticket = REFERENCIA - pd.Timedelta(minutes=PARAMETROS["janela_ticket_min"])
na_janela = F.col("dt_local") > F.lit(inicio_ticket.to_pydatetime())
ticket = (base.groupBy("metodo_pagamento")
          .agg(F.sum(F.when(na_janela, 1).otherwise(0)).alias("pedidos"),
               F.sum(F.when(na_janela, F.col("valor_total"))).alias("receita"),
               F.sum(F.when(~na_janela, 1).otherwise(0)).alias("pedidos_historico"),
               F.sum(F.when(~na_janela, F.col("valor_total"))).alias("receita_historico")).toPandas())
ticket["ticket_medio"] = [razao(r, p) for r, p in zip(ticket["receita"], ticket["pedidos"])]
ticket["media_historica"] = [razao(r, p) for r, p in zip(ticket["receita_historico"], ticket["pedidos_historico"])]
ticket["desvio_pct"] = [None if t is None or not m else round((t / m - 1) * 100, 1) for t, m in zip(ticket["ticket_medio"], ticket["media_historica"])]
ticket["janela_inicio"], ticket["janela_fim"] = inicio_ticket, REFERENCIA
ticket["situacao"] = ["nao_monitorado" if m != PARAMETROS["metodo_monitorado"] else "sem_pedidos" if not p else "sem_historico" if d is None else
                      "alerta" if d < -PARAMETROS["queda_pix_pct"] else "normal"
                      for m, p, d in zip(ticket["metodo_pagamento"], ticket["pedidos"], ticket["desvio_pct"])]
ticket = ticket[["metodo_pagamento", "janela_inicio", "janela_fim", "pedidos", "receita", "ticket_medio", "media_historica", "desvio_pct", "situacao"]]
ticket[["receita", "ticket_medio", "media_historica"]] = ticket[["receita", "ticket_medio", "media_historica"]].astype(float).round(2)

# ------------------------------ validação ------------------------------
com_pedidos = ticket[ticket["pedidos"] > 0]
checar(((com_pedidos["ticket_medio"] * com_pedidos["pedidos"] - com_pedidos["receita"]).abs() <= 0.01 * com_pedidos["pedidos"]).all(),
       "Ticket × pedidos = receita em todos os métodos", "Ticket incoerente com a receita")
checar(PARAMETROS["metodo_monitorado"] in set(ticket["metodo_pagamento"]), "Pix presente na Silver",
       "Nenhum pedido Pix na Silver: a regra 7 não tem o que monitorar", parar=False)

# ------------------------------ resultado ------------------------------
display(ticket)

✅ Ticket × pedidos = receita em todos os métodos
✅ Pix presente na Silver


metodo_pagamento,janela_inicio,janela_fim,pedidos,receita,ticket_medio,media_historica,desvio_pct,situacao
boleto,2026-10-07T09:55:55.051Z,2026-10-07T10:55:55.051Z,16,3696.2,231.01,269.56,-14.3,nao_monitorado
cartão débito,2026-10-07T09:55:55.051Z,2026-10-07T10:55:55.051Z,23,4337.04,188.57,307.41,-38.7,nao_monitorado
cartão crédito,2026-10-07T09:55:55.051Z,2026-10-07T10:55:55.051Z,27,7091.27,262.64,281.92,-6.8,nao_monitorado
pix,2026-10-07T09:55:55.051Z,2026-10-07T10:55:55.051Z,22,4891.82,222.36,261.17,-14.9,normal


## Etapa 5 — Regra 8 · taxa de cancelamento em 30 minutos
**Faz:** % de pedidos `CANCELADO` sobre o total em janelas **fixas** de 30 min das últimas horas (a planilha só pede janela deslizante na regra 6).
Situação: `aberta` (dentro do *watermark*) ou, já fechada, `alerta` (taxa > 10%) ou `normal`. Toda janela fechada vale, qualquer que seja o número de pedidos.
**Valida:** cancelados ≤ pedidos em toda janela.

In [0]:
inicio_canc = REFERENCIA - pd.Timedelta(hours=PARAMETROS["horas_placar_cancelamento"])
cancelamento = (base.filter(F.col("dt_local") > F.lit(inicio_canc.to_pydatetime()))
                .groupBy(F.window("dt_local", f"{PARAMETROS['janela_cancelamento_min']} minutes").alias("janela"))
                .agg(F.count("*").alias("pedidos"), F.sum((F.col("status_pedido") == STATUS_CANCELADO).cast("int")).alias("cancelados"))
                .select(F.col("janela.start").alias("janela_inicio"), F.col("janela.end").alias("janela_fim"), "pedidos", "cancelados")
                .orderBy("janela_inicio").toPandas())
cancelamento["taxa_pct"] = [round(razao(c, p) * 100, 2) for c, p in zip(cancelamento["cancelados"], cancelamento["pedidos"])]
cancelamento["fechada"] = cancelamento["janela_fim"] <= REFERENCIA - WATERMARK
cancelamento["situacao"] = [("aberta" if not f else "alerta" if t > PARAMETROS["cancelamento_max_pct"] else "normal")
                            for f, t in zip(cancelamento["fechada"], cancelamento["taxa_pct"])]

# ------------------------------ validação ------------------------------
checar((cancelamento["cancelados"] <= cancelamento["pedidos"]).all(), f"{len(cancelamento)} janelas coerentes (cancelados ≤ pedidos)",
       "Janela com mais cancelados do que pedidos")

# ------------------------------ resultado ------------------------------
display(cancelamento.sort_values("janela_inicio", ascending=False)) if len(cancelamento) else print("Nenhuma janela de 30 min no período.")

✅ 4 janelas coerentes (cancelados ≤ pedidos)


janela_inicio,janela_fim,pedidos,cancelados,taxa_pct,fechada,situacao
2026-10-07T10:30:00.000Z,2026-10-07T11:00:00.000Z,48,2,4.17,false,aberta
2026-10-07T10:00:00.000Z,2026-10-07T10:30:00.000Z,36,1,2.78,true,normal
2026-10-07T09:30:00.000Z,2026-10-07T10:00:00.000Z,80,3,3.75,true,normal
2026-10-07T09:00:00.000Z,2026-10-07T09:30:00.000Z,39,0,0.0,true,normal


## Etapa 6 — Regra 9 · receita acumulada no dia
**Faz:** soma de `valor_total` desde a meia-noite (Brasília), com cancelados e sem frete, como na planilha (decisão confirmada com o Geovany).
No dia de referência, a comparação é com o **mesmo dia da semana anterior até a mesma hora**; nos dias passados, dia inteiro contra dia inteiro.
`alerta` se a variação passar de ±30%; sem o mesmo dia da semana anterior, a situação é `sem_base` (não há com o que comparar).
**Valida:** a receita do dia de referência bate com a soma da Silver.

In [0]:
dia_ref = REFERENCIA.normalize()
inicio_dias = dia_ref - pd.Timedelta(days=PARAMETROS["dias_placar_receita"] + 7)
diario = (base.filter(F.col("dt_local") >= F.lit(inicio_dias.to_pydatetime()))
          .groupBy(F.to_date("dt_local").alias("data")).agg(F.sum("valor_total").alias("receita_acumulada")).toPandas())
diario["data"] = pd.to_datetime(diario["data"])
semana_anterior_ate_hora = base.filter((F.to_date("dt_local") == F.lit((dia_ref - pd.Timedelta(days=7)).date()))
                                       & (F.col("dt_local") <= F.lit((REFERENCIA - pd.Timedelta(days=7)).to_pydatetime()))) \
                               .agg(F.count("*").alias("n"), F.sum("valor_total").alias("receita")).first()

totais = diario.set_index("data")["receita_acumulada"]
receita = diario[diario["data"] > dia_ref - pd.Timedelta(days=PARAMETROS["dias_placar_receita"])].copy()
receita["atualizado_ate"] = [REFERENCIA if d == dia_ref else d + pd.Timedelta(days=1, seconds=-1) for d in receita["data"]]
receita["receita_semana_anterior"] = [(semana_anterior_ate_hora["receita"] if semana_anterior_ate_hora["n"] else None) if d == dia_ref
                                      else totais.get(d - pd.Timedelta(days=7)) for d in receita["data"]]
receita["variacao_pct"] = [None if not b else round((r / b - 1) * 100, 1) for r, b in zip(receita["receita_acumulada"], receita["receita_semana_anterior"])]
receita["situacao"] = ["sem_base" if v is None else "alerta" if abs(v) > PARAMETROS["variacao_receita_pct"] else "normal" for v in receita["variacao_pct"]]
receita[["receita_acumulada", "receita_semana_anterior"]] = receita[["receita_acumulada", "receita_semana_anterior"]].astype(float).round(2)
receita = receita.sort_values("data", ascending=False)

# ------------------------------ validação ------------------------------
receita_ref_silver = base.filter(F.to_date("dt_local") == F.lit(dia_ref.date())).agg(F.sum("valor_total")).first()[0] or 0
checar(abs(float(receita.loc[receita["data"] == dia_ref, "receita_acumulada"].sum()) - receita_ref_silver) <= 0.01,
       "Receita do dia de referência igual à soma da Silver", "Receita do dia diverge da Silver")

# ------------------------------ resultado ------------------------------
display(receita)

✅ Receita do dia de referência igual à soma da Silver


data,receita_acumulada,atualizado_ate,receita_semana_anterior,variacao_pct,situacao
2026-10-07T00:00:00.000Z,53675.91,2026-10-07T10:55:55.051Z,null,null,sem_base
2026-10-05T00:00:00.000Z,30318.54,2026-10-05T23:59:59.000Z,null,null,sem_base
2026-10-03T00:00:00.000Z,32117.77,2026-10-03T23:59:59.000Z,null,null,sem_base
2026-10-01T00:00:00.000Z,187632.2,2026-10-01T23:59:59.000Z,null,null,sem_base


## Etapa 7 — Alertas
**Faz:** reúne num único mural (`gold_alertas`) os alertas de todas as regras que pedem alerta. Cada alerta tem uma **chave única**: o mesmo alerta nunca é publicado duas vezes.

| Regra | Quando alerta | Um alerta por |
|---|---|---|
| 5 | pedido com status fora da lista (reprovado na Silver) | micro-lote, no ciclo em que ele chega ("alerta imediato"), com os `id_pedido` |
| 6 | menos de X pedidos em janela fechada no pico | episódio: minutos seguidos abaixo de X são o mesmo alerta, mesmo quando a pausa é maior que o placar |
| 7 | Pix mais de 30% abaixo da média histórica | hora |
| 8 | taxa > 10% em janela fechada | janela |
| 9 | receita fora de ±30% | dia e hora |
| 10 | atualização antes do pedido (reprovado na Silver) | micro-lote, no ciclo em que ele chega, com os `id_pedido` |

**Valida:** chaves únicas no ciclo e quantos alertas são novos.

In [0]:
alertas = []
def alertar(regra, indicador, chave, mensagem, valor, limite):
    alertas.append({"ciclo_id": CICLO_ID, "gerado_em_utc": CICLO_EM.replace(tzinfo=None), "regra": regra, "indicador": indicador,
                    "chave": f"{regra}|{chave}", "mensagem": mensagem, "valor": str(valor), "limite": str(limite)})

controle = ler_metadata(ANOTACAO_CONTROLE) or {}

# Regra 6: minutos consecutivos em alerta são o MESMO episódio; um alerta por episódio, com início, fim e o pior volume.
# Um episódio que começa até 1 min depois da última janela já avisada é continuação: numa pausa mais longa que o placar,
# o início visível do episódio anda a cada ciclo, e sem essa memória cada ciclo publicaria o mesmo episódio de novo
ultimo_avisado = pd.to_datetime(controle.get("regra6_ultima_janela_avisada"))
novo_ultimo_avisado = ultimo_avisado
em_alerta = volume[volume["situacao"] == "alerta"].sort_values("janela_inicio")
episodio = (em_alerta["janela_inicio"].diff() > pd.Timedelta(minutes=1)).cumsum() if len(em_alerta) else []
for _, ep in (em_alerta.groupby(episodio) if len(em_alerta) else []):
    continuacao = pd.notna(ultimo_avisado) and ep["janela_inicio"].min() <= ultimo_avisado + pd.Timedelta(minutes=1)
    if not continuacao:
        alertar(6, "gold_kpi_volume_pedidos_5min", ep["janela_inicio"].min(), "Volume abaixo de X no horário de pico",
                f"mín. {int(ep['pedidos'].min())} pedidos/5 min, por {len(ep)} min (até {ep['janela_fim'].max():%H:%M})", PARAMETROS["x_regra6"])
    novo_ultimo_avisado = ep["janela_inicio"].max() if pd.isna(novo_ultimo_avisado) else max(novo_ultimo_avisado, ep["janela_inicio"].max())
for _, t in ticket[ticket["situacao"] == "alerta"].iterrows():
    alertar(7, "gold_kpi_ticket_medio_pagamento", f"{t['metodo_pagamento']}|{REFERENCIA:%Y-%m-%d %H}", "Ticket do Pix abaixo da média histórica",
            f"{t['desvio_pct']}%", f"-{PARAMETROS['queda_pix_pct']}%")
for _, c in cancelamento[cancelamento["situacao"] == "alerta"].iterrows():
    alertar(8, "gold_kpi_taxa_cancelamento_30min", c["janela_inicio"], "Cancelamento acima de 10% na janela de 30 min",
            f"{c['taxa_pct']}% ({int(c['cancelados'])} de {int(c['pedidos'])})", f"{PARAMETROS['cancelamento_max_pct']}%")
for _, r in receita[receita["situacao"] == "alerta"].iterrows():
    alertar(9, "gold_kpi_receita_dia", f"{r['data']:%Y-%m-%d}|{r['atualizado_ate']:%H}", "Receita fora do padrão da semana anterior",
            f"{r['variacao_pct']}%", f"±{PARAMETROS['variacao_receita_pct']}%")

# Regras 5 e 10: a Silver já impediu a persistência; aqui o aviso imediato, um por micro-lote chegado desde o último ciclo
AVISO_QUARENTENA = {5: "Status fora dos valores permitidos", 10: "Atualização de status anterior ao pedido"}
marca_quarentena = nova_marca_quarentena = controle.get("marca_quarentena")
if tabela_delta_existe(CAMINHO_QUARANTINE):
    quarentena = ler_delta(CAMINHO_QUARANTINE)
    if marca_quarentena:
        quarentena = quarentena.filter(F.col("silver_processed_at") > F.to_timestamp(F.lit(marca_quarentena)))
    viola = lambda r: F.col("motivo_quarentena").contains(f"regra{r}_")
    por_lote = (quarentena.groupBy("bronze_source_file")
                .agg(*[F.sum(viola(r).cast("int")).alias(f"n{r}") for r in AVISO_QUARENTENA],
                     *[F.slice(F.sort_array(F.collect_list(F.when(viola(r), F.col("id_pedido")))), 1, 10).alias(f"ids{r}") for r in AVISO_QUARENTENA],
                     F.max("silver_processed_at").alias("marca")).toPandas())
    for _, lote_q in por_lote.iterrows():
        for r, mensagem in AVISO_QUARENTENA.items():
            if lote_q[f"n{r}"]:
                ids = ", ".join(str(i) for i in lote_q[f"ids{r}"])
                alertar(r, "quarantine", lote_q["bronze_source_file"], mensagem, f"{int(lote_q[f'n{r}'])} pedidos (id_pedido: {ids})", "0")
    nova_marca_quarentena = str(por_lote["marca"].max()) if len(por_lote) else marca_quarentena

alertas = pd.DataFrame(alertas, columns=["ciclo_id", "gerado_em_utc", "regra", "indicador", "chave", "mensagem", "valor", "limite"])
caminho_alertas = caminho_gold("gold_alertas")
publicados = ({r["chave"] for r in ler_delta(caminho_alertas).select("chave").collect()} if tabela_delta_existe(caminho_alertas) else set())
alertas_novos = alertas[~alertas["chave"].isin(publicados)]

# ------------------------------ validação ------------------------------
checar(alertas["chave"].is_unique, "Chaves de alerta únicas no ciclo", "Alerta duplicado no ciclo")
checar(alertas_novos.empty, "Nenhum alerta novo", f"{len(alertas_novos)} alertas novos: {alertas_novos['regra'].value_counts().to_dict()}", parar=False)

# ------------------------------ resultado ------------------------------
# display() converte para Spark, que não monta tabela vazia sem tipos: sem linhas, só a frase
display(alertas_novos) if len(alertas_novos) else print("Mural sem alertas novos neste ciclo.")

✅ Chaves de alerta únicas no ciclo
✅ Nenhum alerta novo
Mural sem alertas novos neste ciclo.


## Etapa 8 — Gravação Delta
**Faz:** KPIs em `overwrite`, como pede o Passo 7 (substituem a tabela inteira, inclusive o schema, com `overwriteSchema`); históricos e alertas em `append`
(mantêm as colunas que a tabela já tem, porque o Delta recusa colunas novas em append). Os históricos de janela recebem só janelas **fechadas** ainda
não gravadas; os de ticket e receita recebem uma fotografia por ciclo. A marca da quarentena só avança depois das gravações.
**Valida:** nenhuma janela e nenhum ciclo duplicados nos históricos.

In [0]:
def gravar(nome, pdf, modo):
    """Grava uma tabela da Gold em Delta; tabela vazia é pulada (não há o que gravar)."""
    if modo == "append" and tabela_delta_existe(caminho_gold(nome)):
        # O Delta recusa colunas novas em append (e, com Table ACLs, a migração automática é bloqueada):
        # o histórico mantém as colunas que a tabela já tem
        existentes = ler_delta(caminho_gold(nome)).columns
        pdf = pdf[[c for c in existentes if c in pdf.columns]]
    df = para_spark(pdf)
    if df is None:
        return 0
    if modo == "overwrite":
        # overwrite substitui a tabela inteira, schema incluído: sem overwriteSchema o Delta recusa quando o placar ganha uma coluna
        df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").options(**adls_options).save(caminho_gold(nome))
    else:
        escrever_delta(df, caminho_gold(nome), modo)
    return len(pdf)


def ainda_nao_gravadas(nome, pdf, coluna):
    """Linhas cujo valor em `coluna` ainda não está no histórico (evita repetir janelas entre ciclos)."""
    if pdf.empty or not tabela_delta_existe(caminho_gold(nome)):
        return pdf
    existentes = {r[0] for r in ler_delta(caminho_gold(nome)).select(coluna).distinct().collect()}
    return pdf[~pdf[coluna].isin(existentes)]


com_ciclo = lambda pdf: pdf.assign(ciclo_id=CICLO_ID)
gravadas = {
    "gold_kpi_volume_pedidos_5min": gravar("gold_kpi_volume_pedidos_5min", com_ciclo(volume), "overwrite"),
    "gold_kpi_ticket_medio_pagamento": gravar("gold_kpi_ticket_medio_pagamento", com_ciclo(ticket), "overwrite"),
    "gold_kpi_taxa_cancelamento_30min": gravar("gold_kpi_taxa_cancelamento_30min", com_ciclo(cancelamento), "overwrite"),
    "gold_kpi_receita_dia": gravar("gold_kpi_receita_dia", com_ciclo(receita), "overwrite"),
    "gold_hist_volume_pedidos_5min": gravar("gold_hist_volume_pedidos_5min",
        ainda_nao_gravadas("gold_hist_volume_pedidos_5min", com_ciclo(volume[volume["fechada"]]), "janela_inicio"), "append"),
    "gold_hist_taxa_cancelamento_30min": gravar("gold_hist_taxa_cancelamento_30min",
        ainda_nao_gravadas("gold_hist_taxa_cancelamento_30min", com_ciclo(cancelamento[cancelamento["fechada"]]), "janela_inicio"), "append"),
    "gold_hist_ticket_medio_pagamento": gravar("gold_hist_ticket_medio_pagamento", com_ciclo(ticket), "append"),
    "gold_hist_receita_dia": gravar("gold_hist_receita_dia", com_ciclo(receita[receita["data"] == dia_ref]), "append"),
    "gold_alertas": gravar("gold_alertas", alertas_novos, "append"),
}
escrever_metadata(ANOTACAO_CONTROLE, {"ultimo_ciclo_id": CICLO_ID, "referencia": str(REFERENCIA), "marca_quarentena": nova_marca_quarentena,
                                      "regra6_ultima_janela_avisada": None if pd.isna(novo_ultimo_avisado) else str(novo_ultimo_avisado)})

# ------------------------------ validação ------------------------------
for nome, coluna in [("gold_hist_volume_pedidos_5min", "janela_inicio"), ("gold_hist_taxa_cancelamento_30min", "janela_inicio")]:
    if tabela_delta_existe(caminho_gold(nome)):
        h = ler_delta(caminho_gold(nome))
        checar(h.count() == h.select(coluna).distinct().count(), f"{nome}: nenhuma janela repetida", f"{nome}: janelas repetidas")
display(pd.DataFrame(list(gravadas.items()), columns=["tabela", "linhas_gravadas"]))

✅ gold_hist_volume_pedidos_5min: nenhuma janela repetida
✅ gold_hist_taxa_cancelamento_30min: nenhuma janela repetida


tabela,linhas_gravadas
gold_kpi_volume_pedidos_5min,117
gold_kpi_ticket_medio_pagamento,4
gold_kpi_taxa_cancelamento_30min,4
gold_kpi_receita_dia,4
gold_hist_volume_pedidos_5min,0
gold_hist_taxa_cancelamento_30min,0
gold_hist_ticket_medio_pagamento,4
gold_hist_receita_dia,1
gold_alertas,0


## Etapa 9 — Espelho no SQL Server
**Faz:** espelha só as tabelas gravadas neste ciclo: placares substituídos (`overwrite`); históricos e alertas recebem **todos os ciclos que ainda faltam**
no banco, como no projeto do Lucas. Se o banco falhar, a pendência fica anotada em `metadata/` e o próximo ciclo reenvia todas as tabelas.
Uma falha aqui não desfaz a Delta.
**Valida:** contagem no SQL Server = contagem na Delta, tabela por tabela.

In [0]:
def ciclos_no_banco(nome):
    """ciclo_id já presentes numa tabela do SQL Server (conjunto vazio se a tabela ainda não existe)."""
    try:
        return {r["ciclo_id"] for r in ler_sql_query(f"SELECT DISTINCT ciclo_id FROM {nome_tabela_sql(nome)}").collect()}
    except Exception:
        return set()


espelho, pendencias = [], []
espelho_pendente_anterior = bool(controle.get("espelho_pendente"))   # um ciclo anterior não conseguiu espelhar tudo
if SQL_DISPONIVEL:
    for nome in TABELAS:
        # Só tabelas gravadas neste ciclo; se houve pendência antes, todas, para completar o que faltou
        if not tabela_delta_existe(caminho_gold(nome)) or not (gravadas.get(nome) or espelho_pendente_anterior):
            continue
        delta = ler_delta(caminho_gold(nome))
        try:
            if nome.startswith("gold_kpi_"):
                escrever_sql(delta, nome, "overwrite")
            else:
                faltam = delta.filter(~F.col("ciclo_id").isin(list(ciclos_no_banco(nome))))
                if faltam.count():
                    escrever_sql(faltam, nome, "append")
            n_sql = ler_sql_query(f"SELECT COUNT(*) AS n FROM {nome_tabela_sql(nome)}").first()["n"]
            espelho.append((nome, delta.count(), n_sql))
        except Exception as erro:                                # o próximo ciclo tenta de novo
            pendencias.append(f"{nome}: {str(erro)[:120]}")
espelho = pd.DataFrame(espelho, columns=["tabela", "linhas_delta", "linhas_sql"])
escrever_metadata(ANOTACAO_CONTROLE, {**(ler_metadata(ANOTACAO_CONTROLE) or {}), "espelho_pendente": (not SQL_DISPONIVEL) or bool(pendencias)})

# ------------------------------ validação ------------------------------
if SQL_DISPONIVEL:
    checar(not pendencias, "Espelho concluído em todas as tabelas", f"Pendências no SQL Server: {pendencias}", parar=False)
    checar((espelho["linhas_delta"] == espelho["linhas_sql"]).all(), "Contagem SQL Server = Delta em todas as tabelas",
           f"Divergências: {espelho[espelho['linhas_delta'] != espelho['linhas_sql']]['tabela'].tolist()}", parar=False)
    display(espelho) if len(espelho) else print("Nenhuma tabela espelhada neste ciclo.")
else:
    print("SQL Server indisponível neste ciclo: espelho pendente")

✅ SQL Server: squad2.gold_kpi_volume_pedidos_5min gravada (overwrite)
✅ SQL Server: squad2.gold_kpi_ticket_medio_pagamento gravada (overwrite)
✅ SQL Server: squad2.gold_kpi_taxa_cancelamento_30min gravada (overwrite)
✅ SQL Server: squad2.gold_kpi_receita_dia gravada (overwrite)
✅ SQL Server: squad2.gold_hist_ticket_medio_pagamento gravada (append)
✅ SQL Server: squad2.gold_hist_receita_dia gravada (append)
✅ Espelho concluído em todas as tabelas
✅ Contagem SQL Server = Delta em todas as tabelas


tabela,linhas_delta,linhas_sql
gold_kpi_volume_pedidos_5min,117,117
gold_kpi_ticket_medio_pagamento,4,4
gold_kpi_taxa_cancelamento_30min,4,4
gold_kpi_receita_dia,4,4
gold_hist_ticket_medio_pagamento,136,136
gold_hist_receita_dia,34,34


## Etapa 10 — Resumo para o orquestrador
**Faz:** libera a memória e devolve o resumo do ciclo em JSON (`dbutils.notebook.exit`).

In [0]:
resumo = {
    "ciclo_id": CICLO_ID, "referencia": str(REFERENCIA), "alertas_novos": int(len(alertas_novos)),
    "alertas_por_regra": {int(k): int(v) for k, v in alertas_novos["regra"].value_counts().items()},
    "sql_espelhado": bool(SQL_DISPONIVEL and not pendencias), "tabelas_gravadas": {k: v for k, v in gravadas.items() if v},
}
for _v in ["base", "quarentena", "delta", "faltam"]:
    globals().pop(_v, None)
globals().pop("_v", None)

print(json.dumps(resumo, ensure_ascii=False, default=str))
dbutils.notebook.exit(json.dumps(resumo, ensure_ascii=False, default=str))